# Food Delivery Time Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
df = pd.read_csv("Food_Delivery_Time_Prediction.csv")
df.head()

In [ ]:
# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

for col in df.select_dtypes(include='object'):
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
# Encode categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])

In [ ]:
# Feature Engineering: Haversine Distance
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371
    return c * r

df['Distance_km'] = df.apply(lambda x: haversine(x['Restaurant_Latitude'], x['Restaurant_Longitude'],
                                                x['Delivery_Latitude'], x['Delivery_Longitude']), axis=1)


In [ ]:
# Create Target Variable
df['Delayed'] = df['Delivery_Time'] > df['Delivery_Time'].median()
df['Delayed'] = df['Delayed'].astype(int)

In [ ]:
# Scaling
scaler = StandardScaler()
df[['Distance_km','Delivery_Time']] = scaler.fit_transform(df[['Distance_km','Delivery_Time']])

In [ ]:
# Split Data
X = df.drop(['Delayed'], axis=1)
y = df['Delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [ ]:
# Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

In [ ]:
# KNN with Hyperparameter tuning
param_grid = {'n_neighbors': range(3,10)}
knn = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)

print("Best K:", knn.best_params_)
print("KNN Accuracy:", accuracy_score(y_test, knn_pred))
print(classification_report(y_test, knn_pred))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=5)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

In [ ]:
# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, dt_pred), annot=True, fmt='d')
plt.title("Decision Tree Confusion Matrix")
plt.show()

## Insights

- Naive Bayes works well for simple distributions but may underperform with complex data.
- KNN gives better accuracy but is computationally expensive.
- Decision Tree provides interpretability and balanced performance.

### Recommendation:
Use Decision Tree for explainability or KNN for higher accuracy depending on dataset.
